# Lab 02 — One Bell state, multiple execution targets

Prepare |Φ⁺⟩ = (|00⟩ + |11⟩)/√2, measure and compare across Qiskit Aer CPU/GPU, PennyLane Lightning CPU/GPU, and optionally real IBM QPUs. Start locally: no credentials are needed.

**Physics:** Z-basis measurements should be near 50% 00 and 50% 11. This demonstrates Z correlations, *not* entanglement certification; a later exercise measures X-basis parity and CHSH.

## 1. Choose a backend

| Target | Where | Prerequisite |
| --- | --- | --- |
| `aer` | CPU | Base install |
| `lightning-cpu` | CPU | `pip install -e '.[lightning-cpu]'` |
| `aer-gpu` | A100/H200 runner | GPU-enabled Aer installation |
| `lightning-gpu` | A100/H200 runner | `pip install -e '.[lightning-gpu]'`, CUDA/cuQuantum |
| `ibm` | Physical IBM QPU | Account, quota, explicit confirmation |
| `qrmi` | Slurm QPU allocation | QRMI, injected resource credentials, confirmation |

Changing a notebook selector does not create a GPU allocation. Use the approved GPU runner/Slurm allocation. The two-qubit test verifies execution, not GPU speedup.

In [ ]:
EXECUTOR = "aer"  # Start with CPU. Then try lightning-cpu, GPU, ibm or qrmi.
SHOTS = 1024
SEED = 42
MINIMUM_CORRELATION = 0.70
CONFIRM_REMOTE_QPU_EXECUTION = False  # Mandatory explicit opt-in for real QPUs
BACKEND = None  # Optional IBM backend name; leave None for automatic selection

## 2. Run the *same* workflow as the CLI

Runs are written under `results/` with `manifest.json`, `summary.json`, and `data/counts.json`. QPU access can consume limited quota or incur charges, so remote submission is disabled until you explicitly approve it.

In [ ]:
from pathlib import Path
from quantum_workflows.hello import run_hello

if EXECUTOR in {"ibm", "qrmi"} and not CONFIRM_REMOTE_QPU_EXECUTION:
    raise RuntimeError("Remote QPU is disabled. Check quota/costs and explicitly confirm.")

summary, result_directory = run_hello(
    executor=EXECUTOR,
    shots=SHOTS,
    seed=SEED,
    minimum_correlation=MINIMUM_CORRELATION,
    output=Path("results"),
    backend=BACKEND,
)
print("Executor:", summary["executor"])
print("Backend:", summary["backend"])
print("Provider job:", summary["provider_job_id"] or "local simulation")
print("Bell correlation:", round(summary["bell_correlation"], 4))
print("Results:", result_directory)

## 3. Analyze the physics

The ideal distribution has P(00)=P(11)=1/2 and P(01)=P(10)=0. Estimate ⟨Z⊗Z⟩ = P(same) − P(different) and compare results across targets.

In [ ]:
counts = summary["counts"]
observed = sum(counts.values())
probabilities = {bits: counts.get(bits, 0) / observed for bits in ("00", "01", "10", "11")}
zz = probabilities["00"] + probabilities["11"] - probabilities["01"] - probabilities["10"]
print("Counts:", counts)
print("Probabilities:", {k: round(v, 4) for k, v in probabilities.items()})
print("<Z tensor Z>:", round(zz, 4))
print("Acceptance:", "PASS" if summary["passed"] else "FAIL")
assert observed == SHOTS, "Shot count differs from requested shots"

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Optional chart: pip install matplotlib")
else:
    fig, ax = plt.subplots()
    ax.bar(list(probabilities), list(probabilities.values()))
    ax.set_ylim(0, 1)
    ax.set_ylabel("Probability")
    ax.set_xlabel("Measured outcome")
    ax.set_title(f"Bell state — {EXECUTOR}, {SHOTS} shots")
    plt.show()

## 4. Preserve scientific provenance

Keep the manifest and run directory rather than relying on transient cell output. Compare backend and package versions, source revision, circuit metadata and duration.

In [ ]:
import json
manifest = json.loads((result_directory / "manifest.json").read_text())
print("Run ID:", manifest["experiment_id"])
print("Commit:", manifest["source"].get("commit"))
print("Status:", manifest["status"])
print("Duration (s):", manifest["timings_seconds"].get("total"))
print("Artifacts:", [item["path"] for item in manifest["artifacts"]])

## 5. Advance the experiment

Rerun with `lightning-cpu` to compare SDKs. On a separately authorized A100/H200 runner, choose `aer-gpu` or `lightning-gpu`. After verifying an IBM account and limits, opt into `ibm` execution.

Next: X-basis parity and CHSH; noisy simulations and error mitigation; larger Ising/Hamiltonian dynamics. Analog neutral atoms, photonics and annealing receive modality-specific notebooks rather than being forced into the same gate circuit.